### Speech Command Recognition with torchaudio

From: https://h-huang.github.io/tutorials/intermediate/speech_command_recognition_with_torchaudio.html

This example will show you how to correctly format an audio dataset and then train/test an audio classifier network on the dataset.

If running on Google Colab, have GPU option available. In the menu tabs, select “Runtime” then “Change runtime type”. In the pop-up that follows, you can choose GPU. After the change, your runtime should automatically restart (which means information from executed cells disappear).

First, import the common torch packages such as torchaudio that can be installed by following the instructions on the website.

In [ ]:
# Uncomment the following line to run in Google Colab

# CPU:
# !pip install torch==2.14.0+cpu torchvision==0.29.0+cpu torchaudio==2.11 -f https://download.pytorch.org/whl/torch_stable.html

# GPU:
# !pip install torch==2.14.0+cu130 torchvision==0.29.0+cu130 torchaudio==2.11.1 -f https://download.pytorch.org/whl/torch_stable.html


In [ ]:
#!pip install torchaudio
#!pip install torchvision
#!pip install soundfile
#!pip install pydub

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import torchvision
# import torchcodec   # not needed: we read the wav files with soundfile (see the dataset cell)
import soundfile as sf

import matplotlib.pyplot as plt
import IPython.display as ipd
from tqdm.notebook import tqdm

In [ ]:
print ("torch: ", torch.__version__)
print ("torchvision: ", torchvision.__version__)
print ("torchaudio: ", torchaudio.__version__)
print ("soundfile: ", sf.__version__)

Verify whether a CUDA GPU is available and select our device. Running the network on a GPU will greatly decrease the training/testing runtime.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

### Importing the Dataset
We use torchaudio to download and represent the dataset. Here we use SpeechCommands, which is a datasets of 35 commands spoken by different people. The dataset SPEECHCOMMANDS is a `torch.utils.data.Dataset` version of the dataset. In this dataset, all audio files are about 1 second long (and so about 16000 time frames long).

The actual loading and formatting steps happen when a data point is being accessed, and torchaudio takes care of converting the audio files to tensors. If one wants to load an audio file directly instead, `torchaudio.load()` can be used. It returns a tuple containing the newly created tensor along with the sampling frequency of the audio file (16kHz for SpeechCommands).

Going back to the dataset, here we create a subclass that splits it into standard training, validation, testing subsets

**Modification:** the dataset is downloaded only the first time. If the folder `./SpeechCommands` already exists, `download=False` is passed, so the 2.3 GB archive is not downloaded again.

**Modifications to the dataset class:**
* `os.path.normpath` fixes a bug in the original tutorial: the paths in the validation/test lists did not match the paths in `_walker` (`./SpeechCommands/...` vs `SpeechCommands/...`), so nothing was excluded and the training set contained all 105,829 files, including every validation and test recording. With the fix the training set has 84,843 files.
* `__getitem__` reads each `.wav` with `soundfile` instead of `torchaudio.load`. Recent torchaudio versions decode audio through `torchcodec`, which needs FFmpeg's shared libraries and fails to load on some installations (`OSError: Could not load this library: ... libtorchcodec_core4.dylib`). It returns exactly the same tuple as before.

In [ ]:
from torchaudio.datasets import SPEECHCOMMANDS
import os

# torchaudio stores the dataset here after the first download (~2.3 GB)
DATA_DIR = "./SpeechCommands"

class SubsetSC(SPEECHCOMMANDS):
    def __init__(self, subset: str = None):
        # download only if the dataset is not already on disk
        super().__init__("./", download=not os.path.isdir(DATA_DIR))

        def load_list(filename):
            filepath = os.path.join(self._path, filename)
            with open(filepath) as fileobj:
                # normpath: _walker stores 'SpeechCommands/...' while os.path.join gives './SpeechCommands/...';
                # without it no file is excluded and the training set also contains the validation and test files
                return [os.path.normpath(os.path.join(self._path, line.strip())) for line in fileobj]

        if subset == "validation":
            self._walker = load_list("validation_list.txt")
        elif subset == "testing":
            self._walker = load_list("testing_list.txt")
        elif subset == "training":
            excludes = load_list("validation_list.txt") + load_list("testing_list.txt")
            excludes = set(excludes)
            self._walker = [w for w in self._walker if os.path.normpath(w) not in excludes]

    def __getitem__(self, n):
        # read the wav file with soundfile instead of torchaudio.load (which needs torchcodec + FFmpeg)
        relpath, _, label, speaker_id, utterance_number = self.get_metadata(n)
        data, sample_rate = sf.read(os.path.join(self._archive, relpath), dtype="float32")
        waveform = torch.from_numpy(data).unsqueeze(0)    # shape (1, samples), like torchaudio.load
        return waveform, sample_rate, label, speaker_id, utterance_number


# Create training, validation and testing splits of the data
train_set = SubsetSC("training")
test_set = SubsetSC("testing")
validation_set = SubsetSC("validation")
print("training:", len(train_set), " validation:", len(validation_set), " testing:", len(test_set))
# expected for SpeechCommands v0.02: training 84843, validation 9981, testing 11005

waveform, sample_rate, label, speaker_id, utterance_number = train_set[0]

A data point in the SPEECHCOMMANDS dataset is a tuple made of a waveform (the audio signal), the sample rate, the utterance (label), the ID of the speaker, the number of the utterance.

In [ ]:
print("Shape of waveform: {}".format(waveform.size()))
print("Sample rate of waveform: {}".format(sample_rate))

plt.plot(waveform.t().numpy());

Let’s find the list of labels available in the dataset.

**Modification:** the original cell below is disabled (commented out) because it reads every training file and takes a long time. The labels are written out explicitly in the cell after it.

In [ ]:
# DISABLED: this cell loads every one of the ~85,000 training files just to collect the labels,
# which is very slow. The labels are listed explicitly in the next cell instead.
# labels = sorted(list(set(datapoint[2] for datapoint in train_set)))
# labels

In [ ]:
# The 35 labels of SpeechCommands v0.02, in the same (sorted) order the disabled cell produced
labels = ['backward', 'bed', 'bird', 'cat', 'dog', 'down', 'eight', 'five', 'follow',
          'forward', 'four', 'go', 'happy', 'house', 'learn', 'left', 'marvin', 'nine',
          'no', 'off', 'on', 'one', 'right', 'seven', 'sheila', 'six', 'stop', 'three',
          'tree', 'two', 'up', 'visual', 'wow', 'yes', 'zero']
print(len(labels), 'labels')

The 35 audio labels are commands that are said by users. The first few files are people saying “backward”.

In [ ]:
waveform_first, *_ = train_set[0]
ipd.Audio(waveform_first.numpy(), rate=sample_rate)

In [ ]:
waveform_second, *_ = train_set[1]
ipd.Audio(waveform_second.numpy(), rate=sample_rate)

The last file is someone saying “zero”.

In [ ]:
waveform_last, *_ = train_set[-1]
ipd.Audio(waveform_last.numpy(), rate=sample_rate)

### Formatting the Data
This is a good place to apply transformations to the data. For the waveform, we downsample the audio for faster processing without losing too much of the classification power.

We don’t need to apply other transformations here. It is common for some datasets though to have to reduce the number of channels (say from stereo to mono) by either taking the mean along the channel dimension, or simply keeping only one of the channels. Since SpeechCommands uses a single channel for audio, this is not needed here.

In [ ]:
new_sample_rate = 8000
transform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=new_sample_rate)
transformed = transform(waveform)

ipd.Audio(transformed.numpy(), rate=new_sample_rate)

We are encoding each word using its index in the list of labels.

In [ ]:
def label_to_index(word):
    # Return the position of the word in labels
    return torch.tensor(labels.index(word))


def index_to_label(index):
    # Return the word corresponding to the index in labels
    # This is the inverse of label_to_index
    return labels[index]


word_start = "yes"
index = label_to_index(word_start)
word_recovered = index_to_label(index)

print(word_start, "-->", index, "-->", word_recovered)

To turn a list of data point made of audio recordings and utterances into two batched tensors for the model, we implement a `collate` function which is used by the PyTorch DataLoader that allows us to iterate over a dataset by batches. Please see the documentation (https://docs.pytorch.org/docs/2.15/data.html#working-with-collate-fn) 

In the collate function, we also apply the resampling, and the text encoding for more information about working with a `collate` function.

In [ ]:
def pad_sequence(batch):
    # Make all tensor in a batch the same length by padding with zeros
    batch = [item.t() for item in batch]
    batch = torch.nn.utils.rnn.pad_sequence(batch, batch_first=True, padding_value=0.)
    return batch.permute(0, 2, 1)


def collate_fn(batch):

    # A data tuple has the form:
    # waveform, sample_rate, label, speaker_id, utterance_number

    tensors, targets = [], []

    # Gather in lists, and encode labels as indices
    for waveform, _, label, *_ in batch:
        tensors += [waveform]
        targets += [label_to_index(label)]

    # Group the list of tensors into a batched tensor
    tensors = pad_sequence(tensors)
    targets = torch.stack(targets)

    return tensors, targets


batch_size = 256

if device == "cuda":
    num_workers = 1
    pin_memory = True
else:
    num_workers = 0
    pin_memory = False

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=num_workers,
    pin_memory=pin_memory,
)
validation_loader = torch.utils.data.DataLoader(
    validation_set,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    collate_fn=collate_fn,
    num_workers=num_workers,
    pin_memory=pin_memory,
)
test_loader = torch.utils.data.DataLoader(
    test_set,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    collate_fn=collate_fn,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

### Define the Network
For this tutorial we will use a convolutional neural network to process the raw audio data. Usually more advanced transforms are applied to the audio data, however CNNs can be used to accurately process the raw data. The specific architecture is modeled after the M5 network architecture described in this paper ( https://arxiv.org/pdf/1610.00087 ). An important aspect of models processing raw audio data is the receptive field of their first layer’s filters. Our model’s first filter is length 80 so when processing audio sampled at 8kHz the receptive field is around 10ms (and at 4kHz, around 20 ms). This size is similar to speech processing applications that often use receptive fields ranging from 20ms to 40ms.

In [ ]:
class M5(nn.Module):
    def __init__(self, n_input=1, n_output=35, stride=16, n_channel=32):
        super().__init__()
        self.conv1 = nn.Conv1d(n_input, n_channel, kernel_size=80, stride=stride)
        self.bn1 = nn.BatchNorm1d(n_channel)
        self.pool1 = nn.MaxPool1d(4)
        self.conv2 = nn.Conv1d(n_channel, n_channel, kernel_size=3)
        self.bn2 = nn.BatchNorm1d(n_channel)
        self.pool2 = nn.MaxPool1d(4)
        self.conv3 = nn.Conv1d(n_channel, 2 * n_channel, kernel_size=3)
        self.bn3 = nn.BatchNorm1d(2 * n_channel)
        self.pool3 = nn.MaxPool1d(4)
        self.conv4 = nn.Conv1d(2 * n_channel, 2 * n_channel, kernel_size=3)
        self.bn4 = nn.BatchNorm1d(2 * n_channel)
        self.pool4 = nn.MaxPool1d(4)
        self.fc1 = nn.Linear(2 * n_channel, n_output)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(self.bn1(x))
        x = self.pool1(x)
        x = self.conv2(x)
        x = F.relu(self.bn2(x))
        x = self.pool2(x)
        x = self.conv3(x)
        x = F.relu(self.bn3(x))
        x = self.pool3(x)
        x = self.conv4(x)
        x = F.relu(self.bn4(x))
        x = self.pool4(x)
        x = F.avg_pool1d(x, x.shape[-1])
        x = x.permute(0, 2, 1)
        x = self.fc1(x)
        return F.log_softmax(x, dim=2)


model = M5(n_input=transformed.shape[0], n_output=len(labels))
model.to(device)
print(model)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


n = count_parameters(model)
print("Number of parameters: %s" % n)

We will use an Adam optimizer with weight decay set to 0.0001. At first, we will train with a learning rate of 0.01, but we will use a scheduler to decrease it to 0.001 during training after 20 epochs.

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=0.0001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)  # reduce the learning after 20 epochs by a factor of 10

### Training and Testing the Network
Now let’s define a training function that will feed our training data into the model and perform the backward pass and optimization steps. For training, the loss we will use is the negative log-likelihood. The network will then be tested after each epoch to see how the accuracy varies during the training.

**Modification:** `train()` now also accumulates the loss and the number of correct predictions over all batches, and returns the epoch's average **training loss** and **training accuracy**.

Note: these are measured while the weights are still being updated during the epoch (the same convention Keras uses). Evaluating the whole training set again after each epoch would be more precise, but it would roughly double the training time.

In [ ]:
def train(model, epoch, log_interval):
    model.train()
    total_loss, correct, count = 0.0, 0, 0
    for batch_idx, (data, target) in enumerate(train_loader):

        data = data.to(device)
        target = target.to(device)

        # apply transform and model on whole batch directly on device
        data = transform(data)
        output = model(data)

        # negative log-likelihood for a tensor of size (batch x 1 x n_output)
        loss = F.nll_loss(output.squeeze(1), target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # accumulate statistics for the whole epoch
        total_loss += loss.item() * len(target)
        correct += number_of_correct(get_likely_index(output), target)
        count += len(target)

        # print training stats
        if batch_idx % log_interval == 0:
            print(f"Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} ({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}")

        # update progress bar
        pbar.update(pbar_update)
        # record loss
        losses.append(loss.item())

    # average loss and accuracy over the epoch
    return total_loss / count, 100. * correct / count

Now that we have a training function, we need one for measuring the network's loss and accuracy on a data set. We set the model to `eval()` mode and run inference. Calling `eval()` sets the training variable in all modules in the network to false. Certain layers like batch normalization and dropout behave differently during training, so this step is crucial for getting correct results.

**Modification:** the original `test(model, epoch)` function is replaced by a general `evaluate(model, loader)` that returns the average loss and the accuracy on any data set. We call it on the **validation set** at the end of every epoch, and on the **test set** once, after training is finished.

In [ ]:
def number_of_correct(pred, target):
    # count number of correct predictions
    return pred.squeeze().eq(target).sum().item()


def get_likely_index(tensor):
    # find most likely label index for each element in the batch
    return tensor.argmax(dim=-1)


def evaluate(model, loader, update_pbar=True):
    # Return the average loss and the accuracy (%) of the model on a data set
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():          # no gradients needed: faster, uses less memory
        for data, target in loader:

            data = data.to(device)
            target = target.to(device)

            # apply transform and model on whole batch directly on device
            data = transform(data)
            output = model(data)

            total_loss += F.nll_loss(output.squeeze(1), target, reduction="sum").item()
            pred = get_likely_index(output)
            correct += number_of_correct(pred, target)

            # update progress bar
            if update_pbar:
                pbar.update(pbar_update)

    n = len(loader.dataset)
    return total_loss / n, 100. * correct / n

Finally, we can train the network. With `n_epoch = 20` the learning rate stays at 0.01 for the whole run (the scheduler lowers it after epoch 20); increase `n_epoch` to see the effect of the lower learning rate.

**Modification:** at the end of every epoch the model is evaluated on the **validation set**. The training and validation loss and accuracy are stored in `history` so we can plot how they evolve.

In [ ]:
log_interval = 20
n_epoch = 20     # the scheduler lowers the learning rate after 20 epochs; use fewer epochs to try things out quickly

pbar_update = 1 / (len(train_loader) + len(validation_loader))
losses = []

# history of the loss and accuracy at the end of every epoch
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

# The transform needs to live on the same device as the model and the data.
transform = transform.to(device)
with tqdm(total=n_epoch) as pbar:
    for epoch in range(1, n_epoch + 1):
        train_loss, train_acc = train(model, epoch, log_interval)
        val_loss, val_acc = evaluate(model, validation_loader)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"\nEpoch {epoch}: train loss {train_loss:.4f}, train accuracy {train_acc:.1f}% | "
              f"validation loss {val_loss:.4f}, validation accuracy {val_acc:.1f}%\n")

# Let's plot the training loss versus the number of iteration.
plt.plot(losses);
plt.title("training loss per batch");
plt.xlabel("iteration");

### Accuracy and loss on the training and validation sets
The first graph shows the evolution of the accuracy on both sets, the second graph the evolution of the loss. If the validation curve stops improving while the training curve keeps improving, the model has started to overfit.

In [ ]:
epochs = range(1, len(history["train_acc"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_acc"], "o-", label="training")
plt.plot(epochs, history["val_acc"], "s-", label="validation")
plt.title("Accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy (%)")
plt.xticks(epochs)
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_loss"], "o-", label="training")
plt.plot(epochs, history["val_loss"], "s-", label="validation")
plt.title("Loss (negative log-likelihood)")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.xticks(epochs)
plt.grid(True)
plt.legend()
plt.show()

### Accuracy of the final model on the test set
The test set was not used for training nor for monitoring during training, so its accuracy is an unbiased estimate of how well the model recognizes commands from new speakers.

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, update_pbar=False)
print(f"Test set: loss {test_loss:.4f}, accuracy {test_acc:.2f}% "
      f"({round(test_acc * len(test_set) / 100)}/{len(test_set)} correct)")

The original tutorial reports that the network should be more than 65% accurate on the test set after 14 epochs, and 85% after 21 epochs. Let’s look at the last words in the train set, and see how the model did on it.

In [ ]:
def predict(tensor):
    # Use the model to predict the label of the waveform
    tensor = tensor.to(device)
    tensor = transform(tensor)
    tensor = model(tensor.unsqueeze(0))
    tensor = get_likely_index(tensor)
    tensor = index_to_label(tensor.squeeze())
    return tensor


waveform, sample_rate, utterance, *_ = train_set[-4343]
ipd.Audio(waveform.numpy(), rate=sample_rate)

print(f"Expected: {utterance}. Predicted: {predict(waveform)}.")

Let’s find an example that isn’t classified correctly, if there is one.

In [ ]:
for i, (waveform, sample_rate, utterance, *_) in enumerate(test_set):
    output = predict(waveform)
    if output != utterance:
        ipd.Audio(waveform.numpy(), rate=sample_rate)
        print(f"Data point #{i}. Expected: {utterance}. Predicted: {output}.")
        break
else:
    print("All examples in this dataset were correctly classified!")
    print("In this case, let's just look at the last data point")
    ipd.Audio(waveform.numpy(), rate=sample_rate)
    print(f"Data point #{i}. Expected: {utterance}. Predicted: {output}.")

### Conclusion
In this tutorial, we used torchaudio to load a dataset and resample the signal. We have then defined a neural network that we trained to recognize a given command. There are also other data preprocessing methods, such as finding the mel frequency cepstral coefficients (MFCC), that can reduce the size of the dataset. This transform is also available in torchaudio as torchaudio.transforms.MFCC.